MC simulation of Liquid Argon

Metropolis Monte Carlo for the same argon system as the MD notebook, starting from the thermalized MD configuration. I compare U, P and E (with autocorrelation corrected errors) to the MD results. Helper functions are in `Sci_comp_helpers.py`.

In [14]:
import numpy as np
from Sci_comp_helpers import pot_energy, unload_from_npz,virial,pressure_from_virial,kin_energy
from Sci_comp_helpers import acf,get_tau_int,norm_covariance


def pair_pot_energy(r2):
    invr2 = 1.0 / r2
    invr6 = invr2**3
    return 4.0 * (invr6**2 - invr6)

def delta_U_one_particle(pos, L, i, new_ri):
    """dU when moving particle i to new_ri; only pairs with i change."""
    d_old = pos[i] - pos
    d_old -= L * np.around(d_old / L)
    r2_old = np.sum(d_old**2, axis=1)
    r2_old[i] = np.inf
    U_old_i = np.sum(pair_pot_energy(r2_old))

    d_new = new_ri - pos
    d_new -= L * np.around(d_new / L)
    r2_new = np.sum(d_new**2, axis=1)
    r2_new[i] = np.inf
    U_new_i = np.sum(pair_pot_energy(r2_new))

    return U_new_i - U_old_i

def delta_K_one_particle(vel, i, new_vi):
    """
    Compute dK when particle i's velocity goes from vel[i] -> to new_vi.
    """
    vi_old = vel[i]
    dK = 24.0 * (np.dot(new_vi, new_vi) - np.dot(vi_old, vi_old))
    return dK

def metropolis_step_velocity(vel, beta, max_dv):
    """
    Single-particle velocity Metropolis update.
    Propose a 3D kick dv ~ U([-max_dv, max_dv]^3) to particle i.
    """
    N = vel.shape[0]
    i = np.random.randint(0, N)
    dv = np.random.uniform(-max_dv, max_dv, size=3)
    new_vi = vel[i] + dv
    dK = delta_K_one_particle(vel, i, new_vi)
    if dK <= 0.0 or np.random.random() < np.exp(-beta * dK):
        vel[i] = new_vi
        return True, dK, vel
    else:
        return False, 0.0, vel

def metropolis_step_position(pos, L, beta, max_dx):
    N = pos.shape[0]
    i = np.random.randint(0, N)
    dr = np.random.uniform(-max_dx, max_dx, size=3)
    new_ri = (pos[i] + dr) % L
    dU = delta_U_one_particle(pos, L, i, new_ri)
    if dU <= 0.0 or np.random.random() < np.exp(-beta * dU):
        pos[i] = new_ri
        return True, dU, pos
    else:
        return False, 0.0, pos


def run_mc(pos, vel,L, T, n_sweeps, max_dx=0.1,max_dv=0.1):

    beta = 1.0 / T
    N = pos.shape[0]
    U_current = pot_energy(pos, L)
    K_current = kin_energy(vel)
    results = {}
    U_series, P_series, E_series = [], [], []
    pos_accepts = 0
    vel_accepts = 0
    for sweep in range(n_sweeps):
        for _ in range(N):
            acc, dU, pos = metropolis_step_position(pos, L, beta, max_dx)
            if acc:
                pos_accepts += 1
                U_current += dU
            acc, dK, vel = metropolis_step_velocity(vel, beta, max_dv)
            if acc:
                vel_accepts += 1
                K_current += dK
            


        G = virial(pos,L)
        P_inst = pressure_from_virial(G, N,L,T)
        E_inst = U_current + K_current

        U_series.append(U_current)
        P_series.append(P_inst)
        E_series.append(E_inst)
    results["U"] = np.array(U_series)
    results["P"] = np.array(P_series)
    results['E'] = np.array(E_series)
    results["pos_acc_rate"] = pos_accepts / (n_sweeps * N)
    results["vel_acc_rate"] = vel_accepts  / (n_sweeps)


    return results

# ac correted stat anylysis
def stats_ac(meas, n_cut_max=200):

    M, A = meas.shape
    mean_sigma = meas.mean(axis=0)
    s = meas.std(axis=0, ddof=1)
    se_naive = s / np.sqrt(M)

    rho_full, _, _ = acf(meas, ncut=M-1) 

    ncut_used = np.empty(A)
    tau_int = np.empty(A)

    for a in range(A):
        r = rho_full[:, a] 
        # where rho first becomes negative          
        rho_zero = np.where(r <= 0.0)[0]     
        if rho_zero.size > 0:
            ncut_z = rho_zero[0] + 1         
        else:
            ncut_z = min(n_cut_max, M - 1)
        ncut_used[a] = ncut_z

        tau_int[a] = get_tau_int(rho_full,ncut=ncut_z)[a]

    se_ac = se_naive * np.sqrt((2.0 * tau_int))

    return {"meas": meas,
        "mean": mean_sigma,       
        "se_naive": se_naive,       
        "tau_int": tau_int,        
        "se_ac": se_ac, 
        "ncut_used": ncut_used,   
        "rho_full": rho_full        
    }

run MC from thermalized pos,vel from saved_file

In [6]:
N = 864
rho = 0.75
L = (N / rho) ** (1/3)
T = 1.074  

file_path = 'pset3/MD_results_for_stat_analysis_2025-10-12_23:09:50.npz'
pos_file, vel_file, steps_file, Forces, PEs, KEs, TEs = unload_from_npz(file_path)
pos_init = pos_file[0]  
vel_init = vel_file[0]
results = run_mc(pos_init, vel_init, L, T, n_sweeps=1000, max_dx=0.1,max_dv=0.1)  # or False




U_mean:  -4437.22822225666
P_mean:  0.8041318578105495
E_mean:  -3049.7273948130783


In [21]:
labels = ["U", "P", "E"]

X_MC = np.column_stack([results[i] for i in labels])
stats = stats_ac(X_MC)
print("ncuts: ", stats["ncut_used"])
print("tau_ints: ", stats["tau_int"])

for i in range(X_MC.shape[1]):
  print(f"{labels[i]}={stats["mean"][i]}+-{stats["se_ac"][i]}")

print("""Pset 2 results:
U =-4448.818095672052+-3.3918304102244976
P  = 0.8030482046635716+-0.0012438981585635128
E =-3056.5910663117684+-0.0002668075379186166

The measured temperature from pset 2 was 1.074 not 1.069 so I used 1.074 here
      """)

ncuts:  [94. 85. 19.]
tau_ints:  [18.52030339 20.86016323  5.35970027]
U=-4437.2282222566555+-5.0396009391372285
P=0.804131857810549+-0.0006371141988917574
E=-3049.727394813082+-3.8554425135654267
Pset 2 results:
U =-4448.818095672052+-3.3918304102244976
P  = 0.8030482046635716+-0.0012438981585635128
E =-3056.5910663117684+-0.0002668075379186166

The measured temperature from pset 2 was 1.074 not 1.069 so I used 1.074 here
      
